# RWJBarnabas Health — Monthly PubMed Publications Finder (v18)

v18: One configurable newsletter title and consistent font/margins for draft and final documents. Existing monthly review, Word, Excel, and versioning workflows are retained.


In [1]:
import re
import time
import calendar
from datetime import date
from pathlib import Path
import xml.etree.ElementTree as ET

import requests
import pandas as pd
from docx import Document
from docx.shared import Inches, Pt

## 1 — Settings: edit **only this cell** each month

- Normally, **only change `REPORT_MONTH`** to `YYYY-MM` (example: September 2026 = `"2026-09"`).
- The Excel and Word files are saved in the notebook working folder (`Path.cwd()`). In VS Code, this is normally the notebook folder. The code prints the exact output path.
- `DATE_TYPE = "edat"` searches articles **entered into PubMed during that calendar month**, which is best for a new-items newsletter. Change to `"pdat"` only if you specifically want articles whose listed **publication date** is in that month; late-indexed records make retrospective updates necessary.
- Optional: enter your work email in `CONTACT_EMAIL`, which NCBI encourages for identification.
- Output files are named with the **run date and version** (for example `2026-10-15_v01`) and will not overwrite earlier runs.

- `NEWSLETTER_TITLE`, `FONT_NAME`, `FONT_SIZE`, and `PAGE_MARGIN_INCHES` control both Word outputs. These usually stay unchanged.


In [2]:
REPORT_MONTH = "2026-09"       # CHANGE THIS ONCE A MONTH

# Newsletter appearance: edit here if you ever want a different title or layout.
NEWSLETTER_TITLE = "RWJBarnabas Health Academic Newsletter"
FONT_NAME = "Aptos"
FONT_SIZE = 10
PAGE_MARGIN_INCHES = 0.8
OUTPUT_FOLDER = Path.cwd()  # Save Word and Excel beside the notebook (VS Code notebook working folder)
DATE_TYPE = "edat"            # "edat" = newly added to PubMed; "pdat" = publication date
CONTACT_EMAIL = ""            # optional: "your.name@hospital.org"

# Include the RWJBarnabas Health system affiliation in the search as well.
INCLUDE_SYSTEM_WIDE_ALIASES = True

# Safety limit: if an unexpectedly huge search occurs, stop rather than omit articles.
MAX_RESULTS = 10000

## 2 — Hospital names and search aliases

The list below includes **the 12 acute-care hospitals, Children's Specialized Hospital, Barnabas Health Medical Group, Rutgers Cancer Institute of New Jersey, and RWJBarnabas Health School of Nursing**, with historical/abbreviated names where useful. Each PubMed query uses the actual affiliation field (`[ad]` is PubMed's shorthand for `[Affiliation]`). The code searches for these phrases **in affiliations**, not everywhere in the article. Some generic names require a New Jersey location for a *confirmed* match.

Feel free to add new verified spelling variants over time. Do not automatically treat every Rutgers affiliation as RWJBH hospital authorship.

In [3]:
# Dictionary: hospital name -> affiliation phrases worth searching.
# These are intentionally editable, not an exhaustive official affiliation registry.
HOSPITAL_ALIASES = {
    "Clara Maass Medical Center": ["Clara Maass Medical Center", "Clara Maass"],
    "Community Medical Center": ["Community Medical Center"],
    "Cooperman Barnabas Medical Center": [
        "Cooperman Barnabas Medical Center", "Saint Barnabas Medical Center",
        "St Barnabas Medical Center", "St. Barnabas Medical Center",
    ],
    "Jersey City Medical Center": ["Jersey City Medical Center"],
    "Monmouth Medical Center": ["Monmouth Medical Center"],
    "Monmouth Medical Center Southern Campus": [
        "Monmouth Medical Center Southern Campus", "Monmouth Medical Center South",
        "Southern Ocean Medical Center", "Kimball Medical Center",
    ],
    "Newark Beth Israel Medical Center": ["Newark Beth Israel Medical Center", "Newark Beth Israel"],
    "Robert Wood Johnson University Hospital": [
        "Robert Wood Johnson University Hospital", "RWJ University Hospital",
    ],
    "Robert Wood Johnson University Hospital Somerset": [
        "Robert Wood Johnson University Hospital Somerset", "Somerset Medical Center",
    ],
    "Robert Wood Johnson University Hospital Hamilton": [
        "Robert Wood Johnson University Hospital Hamilton", "Robert Wood Johnson University Hospital at Hamilton",
    ],
    "Robert Wood Johnson University Hospital Rahway": [
        "Robert Wood Johnson University Hospital Rahway", "Robert Wood Johnson University Hospital at Rahway",
        "Rahway Hospital",
    ],
    "Trinitas Regional Medical Center": ["Trinitas Regional Medical Center", "Trinitas Hospital"],
    "Barnabas Health Medical Group": [
        "Barnabas Health Medical Group", "RWJBarnabas Health Medical Group",
    ],
    "Rutgers Cancer Institute of New Jersey": [
        "Rutgers Cancer Institute of New Jersey", "Rutgers Cancer Institute",
        "Cancer Institute of New Jersey",
    ],
    "Children's Specialized Hospital": [
        "Children's Specialized Hospital", "Childrens Specialized Hospital",
    ],
    "RWJBarnabas Health School of Nursing": [
        "RWJBarnabas Health School of Nursing",
        "RWJ Barnabas Health School of Nursing",
    ],
}
SYSTEM_ALIASES = ["RWJBarnabas Health", "RWJ Barnabas Health", "RWJBarnabas"]

# Names that are especially ambiguous, without supporting city/location information.
LOCATION_HINTS = {
    "Community Medical Center": ["toms river", "new jersey", " nj ", "nj 087", "rwj"],
    "Monmouth Medical Center": ["long branch", "new jersey", " nj ", "rwj", "barnabas"],
    "Robert Wood Johnson University Hospital": ["new brunswick", "new jersey", " nj ", "rwj", "rutgers"],
    "Robert Wood Johnson University Hospital Somerset": ["somerville", "somerset", "new jersey", "rwj"],
    "Robert Wood Johnson University Hospital Hamilton": ["hamilton", "new jersey", "rwj"],
    "Robert Wood Johnson University Hospital Rahway": ["rahway", "new jersey", "rwj"],
    "Monmouth Medical Center Southern Campus": ["lakewood", "new jersey", "rwj", "barnabas"],
}

## 3 — Set the calendar dates and construct the PubMed search

`OR` means **any hospital may match**. `mindate` and `maxdate` limit the specified PubMed date field to the first and last days of your selected month.

In [4]:
year, month = map(int, REPORT_MONTH.split("-"))
if not (1 <= month <= 12):
    raise ValueError("REPORT_MONTH must look like '2026-09'.")
month_start = date(year, month, 1)
month_end = date(year, month, calendar.monthrange(year, month)[1])
OUTPUT_FOLDER.mkdir(parents=True, exist_ok=True)

phrases = sorted(set(
    alias for aliases in HOSPITAL_ALIASES.values() for alias in aliases
) | (set(SYSTEM_ALIASES) if INCLUDE_SYSTEM_WIDE_ALIASES else set()))
query = " OR ".join(f'"{phrase}"[ad]' for phrase in phrases)
print(f"Reporting period: {month_start} through {month_end}")
print(f"Date type: {DATE_TYPE} | Hospital sites: {len(HOSPITAL_ALIASES)} | Search phrases: {len(phrases)}")
print(f"Folder: {OUTPUT_FOLDER}")
print("Search preview:", query[:400], "..." if len(query)>400 else "")

Reporting period: 2026-09-01 through 2026-09-30
Date type: edat | Hospital sites: 16 | Search phrases: 38
Folder: /Users/richardsavel/Library/CloudStorage/Dropbox/python test files/pubmed newsletter/chat version
Search preview: "Barnabas Health Medical Group"[ad] OR "Cancer Institute of New Jersey"[ad] OR "Children's Specialized Hospital"[ad] OR "Childrens Specialized Hospital"[ad] OR "Clara Maass"[ad] OR "Clara Maass Medical Center"[ad] OR "Community Medical Center"[ad] OR "Cooperman Barnabas Medical Center"[ad] OR "Jersey City Medical Center"[ad] OR "Kimball Medical Center"[ad] OR "Monmouth Medical Center"[ad] OR "Monm ...


## 4 — Search PubMed (E-utilities API)

`requests.get()` sends your affiliation query to PubMed. PubMed returns matching **PMIDs**, which are article ID numbers. We request *every result page*, rather than silently stopping at 1,000 results.

The code pauses briefly between requests and stops if PubMed reports an error.

In [5]:
BASE = "https://eutils.ncbi.nlm.nih.gov/entrez/eutils/"
session = requests.Session()
session.headers.update({"User-Agent": "RWJBH-Monthly-PubMed-Newsletter/1.0 (educational workflow)"})
COMMON_PARAMS = {"tool": "RWJBHMonthlyNewsletter"}
if CONTACT_EMAIL.strip():
    COMMON_PARAMS["email"] = CONTACT_EMAIL.strip()

def get_api(endpoint, params):
    response = session.get(BASE + endpoint, params={**COMMON_PARAMS, **params}, timeout=50)
    response.raise_for_status()
    return response

search_params = {
    "db": "pubmed", "term": query, "datetype": DATE_TYPE,
    "mindate": month_start.strftime("%Y/%m/%d"),
    "maxdate": month_end.strftime("%Y/%m/%d"),
    "retmode": "json", "retmax": 0,
}
search_response = get_api("esearch.fcgi", search_params)
search_data = search_response.json()["esearchresult"]
count = int(search_data["count"])
if count > MAX_RESULTS:
    raise RuntimeError(f"PubMed found {count:,} results; above MAX_RESULTS={MAX_RESULTS:,}. Review query first.")

pmids = []
for start in range(0, count, 500):
    time.sleep(0.4)  # under NCBI's standard request-rate limit
    page = get_api("esearch.fcgi", {**search_params, "retstart": start, "retmax": 500})
    pmids.extend(page.json()["esearchresult"]["idlist"])
pmids = list(dict.fromkeys(pmids))
if len(pmids) != count:
    raise RuntimeError(f"Expected {count} unique PMIDs but retrieved {len(pmids)}; please rerun.")
print(f"PubMed returned {len(pmids):,} unique articles.")

PubMed returned 89 unique articles.


## 5 — Fetch article metadata and identify hospital affiliations

PubMed sends detailed article records in **XML**. We extract titles, authors, journal, publication date, DOI, links, and individual author affiliations. We confirm a hospital only when its name/alias appears in an affiliation (with simple precautions for ambiguous names).

An article can belong to **more than one hospital**. We keep it as one publication and list all matched hospitals.
If PubMed does not return details for a search hit, its PMID will appear in **Needs Review** rather than stopping the notebook.


In [6]:
def element_text(node):
    return "" if node is None else " ".join("".join(node.itertext()).split())

def normalize(value):
    return " " + re.sub(r"[^a-z0-9]+", " ", str(value).casefold()).strip() + " "

def hospital_match(affiliations):
    found, uncertain = set(), set()
    for aff in affiliations:
        raw = normalize(aff)
        for hospital, aliases in HOSPITAL_ALIASES.items():
            # Disambiguate the main campus and the Somerset/Hamilton/Rahway branches.
            # Likewise avoid classifying Southern Campus as Long Branch main campus.
            if hospital == "Monmouth Medical Center" and any(
                key in raw for key in ["southern campus", "southern ocean", "lakewood", "kimball"]
            ):
                continue
            if hospital == "Robert Wood Johnson University Hospital" and any(
                key in raw for key in ["somerset", "somerville", "hamilton", "rahway"]
            ):
                continue
            for alias in aliases:
                if normalize(alias) in raw:
                    hints = LOCATION_HINTS.get(hospital)
                    if hints and not any(normalize(x).strip() in raw for x in hints):
                        uncertain.add(hospital)
                    else:
                        found.add(hospital)
                    break
    return sorted(found), sorted(uncertain - found)

def publication_date(article):
    for path in ["MedlineCitation/Article/ArticleDate", "MedlineCitation/Article/Journal/JournalIssue/PubDate"]:
        item = article.find(path)
        if item is not None:
            bits = [element_text(item.find(x)) for x in ["Year", "Month", "Day"]]
            if any(bits):
                return " ".join(b for b in bits if b)
            medline = element_text(item.find("MedlineDate"))
            if medline:
                return medline
    return ""

def parse_article(article):
    med = article.find("MedlineCitation")
    if med is None:
        return None
    pmid = element_text(med.find("PMID"))
    detail = med.find("Article")
    if detail is None:
        return None
    affiliations = sorted({element_text(x) for x in detail.findall(".//AffiliationInfo/Affiliation") if element_text(x)})
    matched, uncertain = hospital_match(affiliations)
    authors = []
    for a in detail.findall("AuthorList/Author"):
        collective = element_text(a.find("CollectiveName"))
        name = " ".join(filter(None, [element_text(a.find("ForeName")), element_text(a.find("LastName"))]))
        if name or collective:
            authors.append(name or collective)
    doi = next((element_text(x) for x in article.findall(".//ArticleId") if x.get("IdType") == "doi"), "")
    system_hit = any(normalize(alias) in normalize(aff) for aff in affiliations for alias in SYSTEM_ALIASES)
    status = "Matched" if matched else "Needs Review"
    reason = "Hospital name found" if matched else (
        "Ambiguous hospital name / location missing" if uncertain else
        "System-wide RWJBH affiliation; hospital not identified" if system_hit else
        "Search hit, but no hospital affiliation verified"
    )
    return {
        "Status": status, "Reason": reason, "PMID": pmid,
        "Title": element_text(detail.find("ArticleTitle")),
        "Authors": "; ".join(authors),
        "Journal": element_text(detail.find("Journal/Title")),
        "Publication Date": publication_date(article),
        "Matched Hospitals": "; ".join(matched),
        "Possible Hospitals": "; ".join(uncertain),
        "DOI": doi,
        "PubMed Link": f"https://pubmed.ncbi.nlm.nih.gov/{pmid}/",
        "DOI Link": f"https://doi.org/{doi}" if doi else "",
        "Affiliations": " | ".join(affiliations),
    }

rows = []
for start in range(0, len(pmids), 150):
    time.sleep(0.4)
    batch = pmids[start:start + 150]
    response = get_api("efetch.fcgi", {"db": "pubmed", "id": ",".join(batch), "retmode": "xml"})
    root = ET.fromstring(response.content)
    articles = root.findall("PubmedArticle")
    # PubMed can return fewer detailed records than the search ID list.
    # Keep any missing PMIDs in the review spreadsheet instead of crashing.
    returned_pmids = set()
    for a in articles:
        item = parse_article(a)
        if item:
            rows.append(item)
            returned_pmids.add(item["PMID"])
    missing = sorted(set(batch) - returned_pmids)
    for missing_pmid in missing:
        print(f"WARNING: PubMed did not supply details for PMID {missing_pmid}; review this link manually.")
        rows.append({
            "Status": "Needs Review",
            "Reason": "PubMed returned a PMID without article details; check manually",
            "PMID": missing_pmid,
            "Title": "DETAILS UNAVAILABLE — CHECK PUBMED",
            "Authors": "", "Journal": "", "Publication Date": "",
            "Matched Hospitals": "", "Possible Hospitals": "",
            "DOI": "", "PubMed Link": f"https://pubmed.ncbi.nlm.nih.gov/{missing_pmid}/",
            "DOI Link": "", "Affiliations": "",
        })
print(f"Processed {len(rows):,} publication records.")

Processed 89 publication records.


## 6 — Make your DataFrame and review counts

Same pandas operations you've used before: construct the DataFrame, remove duplicates by PMID, and filter into **Matched Articles** and **Needs Review**. No articles are silently dropped because of an ambiguous affiliation.

In [7]:
COLUMNS = ["Status", "Reason", "PMID", "Title", "Authors", "Journal", "Publication Date", "Matched Hospitals", "Possible Hospitals", "DOI", "PubMed Link", "DOI Link", "Affiliations"]
df = pd.DataFrame(rows, columns=COLUMNS)
if len(df) != len(pmids):
    raise RuntimeError("At least one article was not parsed. Review the results before exporting.")
df = df.drop_duplicates(subset="PMID").sort_values(["Matched Hospitals", "Title"], na_position="last").reset_index(drop=True)
matched = df.loc[df["Status"] == "Matched"].copy()
needs_review = df.loc[df["Status"] == "Needs Review"].copy()
print(f"Matched articles: {len(matched)} | Needs review: {len(needs_review)} | Total: {len(df)}")
display(df[["Status", "PMID", "Title", "Matched Hospitals", "Possible Hospitals"]].head(20))

Matched articles: 86 | Needs review: 3 | Total: 89


,Status,PMID,Title,Matched Hospitals,Possible Hospitals
0,Needs Review,30335297,DETAILS UNAVAILABLE — CHECK PUBMED,,
1,Needs Review,42804209,Insurance Coverage and Access Requirements for...,,
2,Needs Review,42806094,The ARCADIA-ZERO registry: two-year outcomes o...,,Community Medical Center
3,Matched,42726909,Validation of the English Proficiency Question...,Barnabas Health Medical Group,
4,Matched,42706537,Clinical and genetic landscape of neuronopathi...,Children's Specialized Hospital,
5,Matched,42755109,Contingency discrimination training and resurg...,Children's Specialized Hospital,
6,Matched,42778186,Pediatric Literacy Promotion for Latino Famili...,Children's Specialized Hospital,
7,Matched,42770154,Prevalence of Error Types of Synthesized Conti...,Children's Specialized Hospital,
8,Matched,42697386,The extinction burst: Effects of alternative r...,Children's Specialized Hospital,
9,Matched,42718514,Understanding the patterns of superficial vein...,Children's Specialized Hospital,


## 7 — Export an Excel workbook (with dropdowns)

The workbook contains **By Hospital**, **Matched Articles**, **Needs Review**, and **All Search Results**. In **Needs Review**, the last two columns are Excel dropdowns: **Include?** (`Yes`/`No`) and **Hospital to Assign** (12 hospitals, Rutgers Cancer Institute, Barnabas Health Medical Group, Multiple Hospitals, Other). A hidden choices sheet supports the dropdowns. Save your edits before running only the final cell.

The filenames include today’s date and a sequential version number, so earlier exports are preserved.


In [8]:
# Sort publication dates newest first, even when PubMed provides only a month or year.
# For month-only entries, assume the first day; for year-only entries, assume January 1.
# Missing or unrecognized dates sort last.
def date_for_sort(value):
    import re
    from datetime import datetime
    s = str(value).strip()
    # e.g. "2026 09 30", "2026 Sep 11", "2026 Sep", "2026"
    parts = s.split()
    if not parts or not re.fullmatch(r"\d{4}", parts[0]):
        return pd.NaT
    year = int(parts[0])
    month = 1
    day = 1
    if len(parts) >= 2:
        mon = parts[1].strip(".,")
        if mon.isdigit():
            month = int(mon)
        else:
            try:
                month = datetime.strptime(mon[:3].title(), "%b").month
            except ValueError:
                return pd.NaT
    if len(parts) >= 3 and parts[2].isdigit():
        day = int(parts[2])
    try:
        return pd.Timestamp(year=year, month=month, day=day)
    except ValueError:
        return pd.NaT

# Give multi-hospital collaborations their own section at the end.
# A publication appears only ONCE there, rather than under every hospital.
by_hospital = matched.copy()
def publication_category(value):
    hospitals = [item.strip() for item in str(value).split(";") if item.strip()]
    return "Multiple Hospitals" if len(set(hospitals)) > 1 else (hospitals[0] if hospitals else "")

by_hospital["Hospital"] = by_hospital["Matched Hospitals"].map(publication_category)
by_hospital = by_hospital[by_hospital["Hospital"].ne("")].copy()
by_hospital["_sort_date"] = by_hospital["Publication Date"].map(date_for_sort)
def section_rank(category):
    if category == "Barnabas Health Medical Group":
        return 1
    if category == "Multiple Hospitals":
        return 2
    if category == "Other":
        return 3
    if category == "Rutgers Cancer Institute of New Jersey":
        return 4
    if category == "RWJBarnabas Health School of Nursing":
        return 5
    return 0
by_hospital["_section_last"] = by_hospital["Hospital"].map(section_rank)
by_hospital = by_hospital.sort_values(
    ["_section_last", "Hospital", "_sort_date", "Title"],
    ascending=[True, True, False, True], na_position="last", kind="stable"
).reset_index(drop=True)
print(f"Organized {len(matched)} unique papers into {len(by_hospital)} publication entries.")


Organized 86 unique papers into 86 publication entries.


In [9]:
# Each Run All creates a NEW numbered pair (Excel + Word), never overwriting old files.
from datetime import date as _today_date
run_day = _today_date.today().isoformat()
for run_number in range(1, 1000):
    run_tag = f"{run_day}_v{run_number:02d}"
    run_stem = f"RWJBH_Publications_{REPORT_MONTH}_{run_tag}"
    if not any(OUTPUT_FOLDER.glob(f"{run_stem}*")):
        break
else:
    raise RuntimeError("Too many runs for this day.")
excel_path = OUTPUT_FOLDER / f"{run_stem}.xlsx"
with pd.ExcelWriter(excel_path, engine="xlsxwriter") as writer:
    # Dropdown choices live in a hidden Excel worksheet.
    # A worksheet range avoids Excel's 255-character limit on inline lists.
    hospital_choices = list(HOSPITAL_ALIASES) + ["Multiple Hospitals", "Other"]
    options_sheet = writer.book.add_worksheet("Dropdown Choices")
    options_sheet.write(0, 0, "Hospital / institution")
    for option_row, label in enumerate(hospital_choices, start=1):
        options_sheet.write(option_row, 0, label)
    options_sheet.hide()
    writer.book.define_name(
        "HospitalOptions", f"='Dropdown Choices'!$A$2:$A${len(hospital_choices)+1}"
    )
    for sheet_name, frame in [
        ("By Hospital", by_hospital.drop(columns=["_sort_date", "_section_last"])),
        ("Matched Articles", matched),
        ("Needs Review", needs_review.assign(**{"Include?": "", "Hospital to Assign": ""})),
        ("All Search Results", df),
    ]:
        frame.to_excel(writer, sheet_name=sheet_name, index=False)
        worksheet = writer.sheets[sheet_name]
        workbook = writer.book
        header_fmt = workbook.add_format({"bold": True, "font_color": "white", "bg_color": "#17365D", "text_wrap": True, "valign": "top"})
        wrap_fmt = workbook.add_format({"text_wrap": True, "valign": "top"})
        export_columns = list(frame.columns)
        for col_idx, col_name in enumerate(export_columns):
            width = {"Title": 58, "Authors": 45, "Journal": 30, "Affiliations": 70,
                     "PubMed Link": 44, "DOI Link": 44, "Matched Hospitals": 37,
                     "Possible Hospitals": 32, "Reason": 42, "Hospital": 40, "Include?": 15, "Hospital to Assign": 45}.get(col_name, 18)
            worksheet.set_column(col_idx, col_idx, width, wrap_fmt)
            worksheet.write(0, col_idx, col_name, header_fmt)
        worksheet.freeze_panes(1, 0)
        worksheet.autofilter(0, 0, max(1, len(frame)), len(export_columns) - 1)
        worksheet.set_zoom(85)
        if sheet_name == "Needs Review":
            # Excel's row/column indices are zero-based. Apply through row 1001,
            # so newly added review rows also get dropdowns.
            yes_col = export_columns.index("Include?")
            hospital_col = export_columns.index("Hospital to Assign")
            worksheet.data_validation(1, yes_col, 1000, yes_col, {
                "validate": "list", "source": ["Yes", "No"],
                "input_title": "Include publication?",
                "input_message": "Choose Yes or No from the dropdown.",
                "error_title": "Select Yes or No",
                "error_message": "Please select Yes or No from the list.",
                "error_type": "stop",
            })
            worksheet.data_validation(1, hospital_col, 1000, hospital_col, {
                "validate": "list", "source": "=HospitalOptions",
                "input_title": "Assign institution",
                "input_message": "Choose an institution, Multiple Hospitals, or Other.",
                "error_title": "Use the dropdown",
                "error_message": "Please select an option from the dropdown.",
                "error_type": "stop",
            })
print("Saved Excel:", excel_path)

Saved Excel: /Users/richardsavel/Library/CloudStorage/Dropbox/python test files/pubmed newsletter/chat version/RWJBH_Publications_2026-09_2026-10-15_v01.xlsx


## 8 — Export a Word newsletter draft

**Organized by hospital/institution (alphabetically), with **Multiple Hospitals** after clinical sites, then newest publication first within each hospital.** Details within an entry have no extra paragraph spacing, and a blank line follows each entry. Papers affiliated with multiple hospitals are repeated under each relevant hospital. Only hospital-matched items appear; verify citations and use Excel's `Needs Review` sheet before distribution.
The Word file includes a centered **Page X of Y** footer.

**New in v13:** PubMed and DOI addresses are saved as real clickable Word hyperlinks, in both draft and final documents.


A **Publications by institution** summary table appears at the top (alphabetical, with Multiple Hospitals last). Each publication is counted once in its reporting category.


In [10]:
word_path = OUTPUT_FOLDER / f"{run_stem}_DRAFT.docx"
doc = Document()
section = doc.sections[0]
section.top_margin = Inches(PAGE_MARGIN_INCHES)
section.bottom_margin = Inches(PAGE_MARGIN_INCHES)
normal = doc.styles["Normal"]
normal.font.name = FONT_NAME
normal.font.size = Pt(FONT_SIZE)
normal.paragraph_format.space_before = Pt(0)
normal.paragraph_format.space_after = Pt(0)
normal.paragraph_format.line_spacing = 1.0

def detail_line(text):
    # Each field occupies its own line without extra space after it.
    p = doc.add_paragraph(text)
    p.paragraph_format.space_before = Pt(0)
    p.paragraph_format.space_after = Pt(0)
    p.paragraph_format.line_spacing = 1.0
    return p

def add_clickable_link(paragraph, label, url):
    """Add a real clickable Word hyperlink (not merely URL text)."""
    from docx.oxml import OxmlElement
    from docx.oxml.ns import qn
    from docx.opc.constants import RELATIONSHIP_TYPE as RT
    relation_id = paragraph.part.relate_to(str(url), RT.HYPERLINK, is_external=True)
    hyperlink = OxmlElement("w:hyperlink")
    hyperlink.set(qn("r:id"), relation_id)
    run = OxmlElement("w:r")
    properties = OxmlElement("w:rPr")
    color = OxmlElement("w:color")
    color.set(qn("w:val"), "0563C1")
    underline = OxmlElement("w:u")
    underline.set(qn("w:val"), "single")
    properties.append(color)
    properties.append(underline)
    run.append(properties)
    text = OxmlElement("w:t")
    text.text = str(label)
    run.append(text)
    hyperlink.append(run)
    paragraph._p.append(hyperlink)

def add_link_line(document, prefix, label, url):
    paragraph = document.add_paragraph()
    paragraph.paragraph_format.space_before = Pt(0)
    paragraph.paragraph_format.space_after = Pt(0)
    paragraph.paragraph_format.line_spacing = 1.0
    paragraph.add_run(prefix)
    add_clickable_link(paragraph, label, url)
    return paragraph

# Overview table: count each publication once in its assigned category.
# Only display institutions with publications this month.
def add_summary_table(document, organized_frame, institution_names):
    from docx.enum.table import WD_TABLE_ALIGNMENT, WD_CELL_VERTICAL_ALIGNMENT
    from docx.enum.text import WD_ALIGN_PARAGRAPH
    from docx.shared import Pt

    counts = organized_frame["Hospital"].value_counts().to_dict()
    clinical = sorted(set(institution_names) - {
        "Barnabas Health Medical Group", "Multiple Hospitals", "Other",
        "Rutgers Cancer Institute of New Jersey", "RWJBarnabas Health School of Nursing"
    })
    categories = clinical + ["Barnabas Health Medical Group", "Multiple Hospitals", "Other",
                             "Rutgers Cancer Institute of New Jersey", "RWJBarnabas Health School of Nursing"]
    title = document.add_paragraph()
    title.add_run("Publications by institution").bold = True
    title.paragraph_format.space_before = Pt(8)
    title.paragraph_format.space_after = Pt(4)

    table = document.add_table(rows=1, cols=2)
    table.style = "Light Shading Accent 1"
    table.alignment = WD_TABLE_ALIGNMENT.CENTER
    table.cell(0, 0).text = "Institution / Category"
    table.cell(0, 1).text = "Articles"
    for category in categories:
        # Only show institutions with at least one publication this month.
        if counts.get(category, 0) == 0:
            continue
        cells = table.add_row().cells
        cells[0].text = category
        cells[1].text = str(counts[category])
    total = table.add_row().cells
    total[0].text = "Total unique publications"
    total[1].text = str(len(organized_frame))
    for cell in table.rows[0].cells + table.rows[-1].cells:
        for paragraph in cell.paragraphs:
            for run in paragraph.runs:
                run.bold = True
    for row in table.rows:
        row.cells[1].vertical_alignment = WD_CELL_VERTICAL_ALIGNMENT.CENTER
        for p in row.cells[1].paragraphs:
            p.alignment = WD_ALIGN_PARAGRAPH.RIGHT
        for cell in row.cells:
            for p in cell.paragraphs:
                p.paragraph_format.space_after = Pt(0)
                p.paragraph_format.space_before = Pt(0)
    below = document.add_paragraph()
    below.paragraph_format.space_after = Pt(6)
    return table

doc.add_heading(NEWSLETTER_TITLE, level=0)
intro = doc.add_paragraph(f"{month_start.strftime('%B %Y')} · DRAFT — For review, not distribution")
intro.paragraph_format.space_after = Pt(8)
intro = doc.add_paragraph(
    "Articles newly entered in PubMed during the selected month (or matching your chosen date setting). "
    "Hospital affiliations were identified from PubMed metadata. Verify all citations and affiliations before distributing."
)
intro.paragraph_format.space_after = Pt(8)
intro = doc.add_paragraph(
    f"{len(matched)} unique matched publications · {len(needs_review)} additional records require review in Excel"
)
intro.paragraph_format.space_after = Pt(12)

# Institutions with zero publications are omitted from the summary table.
summary_institutions = list(HOSPITAL_ALIASES.keys()) + ["Other", "Multiple Hospitals"]
add_summary_table(doc, by_hospital, summary_institutions)

if by_hospital.empty:
    doc.add_paragraph("No hospital-matched publications were found for these settings.")
else:
    for hospital, papers in by_hospital.groupby("Hospital", sort=False):
        heading = doc.add_heading(f"{hospital} ({len(papers)})", level=1)
        heading.paragraph_format.space_before = Pt(12)
        heading.paragraph_format.space_after = Pt(6)
        for idx, (_, r) in enumerate(papers.iterrows(), start=1):
            p = detail_line(f"{idx}. {r['Title']}")
            p.runs[0].bold = True
            detail_line(f"Authors: {r['Authors'] or 'Not listed'}")
            detail_line(f"Journal: {r['Journal']} | Publication date: {r['Publication Date'] or 'Not stated'}")
            add_link_line(doc, "PubMed: ", r["PubMed Link"], r["PubMed Link"])
            if r["DOI"]:
                add_link_line(doc, "DOI: ", r["DOI"], f"https://doi.org/{r['DOI']}")
            # One blank line after each complete entry, NOT between its fields.
            spacer = doc.add_paragraph()
            spacer.paragraph_format.space_after = Pt(0)
            spacer.paragraph_format.line_spacing = 1.0


# Word updates PAGE and NUMPAGES fields when opening/printing the document.
def add_page_number_footer(document):
    from docx.enum.text import WD_ALIGN_PARAGRAPH
    from docx.oxml import OxmlElement
    from docx.oxml.ns import qn
    paragraph = document.sections[0].footer.paragraphs[0]
    paragraph.alignment = WD_ALIGN_PARAGRAPH.CENTER
    paragraph.paragraph_format.space_before = Pt(0)
    paragraph.paragraph_format.space_after = Pt(0)
    paragraph.add_run("Page ")
    def field(name):
        run = paragraph.add_run()
        begin = OxmlElement("w:fldChar")
        begin.set(qn("w:fldCharType"), "begin")
        instr = OxmlElement("w:instrText")
        instr.set(qn("xml:space"), "preserve")
        instr.text = f" {name} "
        separate = OxmlElement("w:fldChar")
        separate.set(qn("w:fldCharType"), "separate")
        placeholder = OxmlElement("w:t")
        placeholder.text = "1"
        end = OxmlElement("w:fldChar")
        end.set(qn("w:fldCharType"), "end")
        for node in (begin, instr, separate, placeholder, end):
            run._r.append(node)
    field("PAGE")
    paragraph.add_run(" of ")
    field("NUMPAGES")

add_page_number_footer(doc)
doc.save(word_path)
print("Saved Word draft:", word_path)
print("\nDONE. Open your output folder:", OUTPUT_FOLDER)


Saved Word draft: /Users/richardsavel/Library/CloudStorage/Dropbox/python test files/pubmed newsletter/chat version/RWJBH_Publications_2026-09_2026-10-15_v01_DRAFT.docx

DONE. Open your output folder: /Users/richardsavel/Library/CloudStorage/Dropbox/python test files/pubmed newsletter/chat version


## 9 — Your monthly checklist

1. Edit `REPORT_MONTH` in section 2, e.g. `"2026-10"` for October 2026.
2. Click **Run All** in VS Code (internet connection required). Check the printed `Folder:` line to see the exact save location.
3. Open the **Excel workbook**, starting with `By Hospital`. Check the matches and review every candidate in `Needs Review`.
4. Open the **Word DRAFT**, now grouped by hospital and sorted newest first, remove false matches, and add any true articles from `Needs Review` that you verified.
5. Copy/edit the Word text for your newsletter. Save the finalized document separately.

**Coverage notes:** `[Affiliation]` works well but not every PubMed record contains every author's current institution. The system-wide organization name is also searched to find candidates, but **Rutgers-only results are not automatically counted**. Historical hospital names are suggestions to validate, not a guarantee of completeness. Rerunning the same calendar month later can return a different set of results if PubMed changes its records. If you plan to publish a digest every month, retain past Excel workbooks for auditing and deduplication.

**Sources:** [RWJBarnabas Health official acute-care facility list](https://www.rwjbh.org/careers/physician-careers/why-choose-rwjbarnabas-health/); [NCBI E-utilities date parameters](https://www.ncbi.nlm.nih.gov/books/NBK25499/).

## 10 — Final Word document after your Excel review

Open the newest numbered Excel file, fill in the **Needs Review** dropdowns, and **save and close Excel**. Then run **only the last cell**. It reads that workbook without another PubMed search, and creates a **new numbered FINAL Word file** each time. Earlier files are retained. The Word document includes **Page X of Y** at the bottom of each page.

**Important:** Keep the notebook in VS Code's current working folder. The last cell reports which Excel file it read; verify its filename if you have several versions. Never use **Run All** after you have edited a workbook unless you intend to generate a fresh, separate run.

**New in v13:** The final report also retains clickable PubMed and DOI links.


The final document also includes an automatically updated summary table based on your Excel review decisions.


**New in v15:** Children's Specialized Hospital and RWJBarnabas Health School of Nursing appear in PubMed searching and the Excel dropdown; the Cancer Institute and RWJBarnabas Health School of Nursing categories appear after the RWJBarnabas clinical categories in the Word digest and summary table.


In [12]:
# FINAL EXPORT — run ONLY this cell after saving and closing the reviewed Excel file.
from pathlib import Path
import pandas as pd
from docx import Document
from docx.shared import Inches, Pt

# This final cell also works when opened in a fresh notebook session.
REPORT_MONTH = globals().get("REPORT_MONTH", "2026-09")
# These defaults let the final cell run on its own in a fresh kernel.
# After running the settings cell, its values are used automatically.
NEWSLETTER_TITLE = globals().get("NEWSLETTER_TITLE", "RWJBarnabas Health Academic Newsletter")
FONT_NAME = globals().get("FONT_NAME", "Aptos")
FONT_SIZE = globals().get("FONT_SIZE", 10)
PAGE_MARGIN_INCHES = globals().get("PAGE_MARGIN_INCHES", 0.8)
OUTPUT_FOLDER = Path(globals().get("OUTPUT_FOLDER", Path.cwd()))
month_start = pd.Timestamp(REPORT_MONTH + "-01")

# Sort publication dates newest first, even when PubMed provides only a month or year.
# For month-only entries, assume the first day; for year-only entries, assume January 1.
# Missing or unrecognized dates sort last.
def date_for_sort(value):
    import re
    from datetime import datetime
    s = str(value).strip()
    # e.g. "2026 09 30", "2026 Sep 11", "2026 Sep", "2026"
    parts = s.split()
    if not parts or not re.fullmatch(r"\d{4}", parts[0]):
        return pd.NaT
    year = int(parts[0])
    month = 1
    day = 1
    if len(parts) >= 2:
        mon = parts[1].strip(".,")
        if mon.isdigit():
            month = int(mon)
        else:
            try:
                month = datetime.strptime(mon[:3].title(), "%b").month
            except ValueError:
                return pd.NaT
    if len(parts) >= 3 and parts[2].isdigit():
        day = int(parts[2])
    try:
        return pd.Timestamp(year=year, month=month, day=day)
    except ValueError:
        return pd.NaT


# Using the same folder and reporting month specified at the top of the notebook.
# Prefer the workbook from the Run All session. If starting a new kernel,
# choose the most recently modified numbered workbook for this month.
if "excel_path" in globals() and Path(excel_path).is_file():
    review_excel = Path(excel_path)
else:
    candidates = list(OUTPUT_FOLDER.glob(f"RWJBH_Publications_{REPORT_MONTH}_????-??-??_v??.xlsx"))
    if not candidates:
        raise FileNotFoundError("No numbered Excel workbook found. Run the notebook to create one first.")
    review_excel = max(candidates, key=lambda f: f.stat().st_mtime)
print(f"Reading reviewed Excel: {review_excel.name}")
original = pd.read_excel(review_excel, sheet_name="Matched Articles", dtype={"PMID": str}).fillna("")
review = pd.read_excel(review_excel, sheet_name="Needs Review", dtype={"PMID": str}).fillna("")

required = {"Include?", "Hospital to Assign"}
if not required.issubset(review.columns):
    raise ValueError("Needs Review must contain columns 'Include?' and 'Hospital to Assign'. Rerun the updated Excel export cell once to create them, BEFORE doing your edits.")

# Accept familiar names, abbreviations, and minor spelling differences.
# Ambiguous or unrecognized names STOP with a helpful message rather than guessing.
import re
from difflib import get_close_matches

def clean_hospital_name(value):
    return re.sub(r"[^a-z0-9]+", " ", str(value).casefold()).strip()

# Manual-entry aliases are intentionally more flexible than the PubMed search aliases.
MANUAL_ALIASES = {
    "RWJ University Hospital New Brunswick": "Robert Wood Johnson University Hospital",
    "RWJ University Hospital": "Robert Wood Johnson University Hospital",
    "RWJUH New Brunswick": "Robert Wood Johnson University Hospital",
    "RWJ New Brunswick": "Robert Wood Johnson University Hospital",
    "Robert Wood Johnson University Hospital New Brunswick": "Robert Wood Johnson University Hospital",
    "Robert Wood Johnson University Hospital at New Brunswick": "Robert Wood Johnson University Hospital",
    "RWJ University Hospital Somerset": "Robert Wood Johnson University Hospital Somerset",
    "RWJ University Hospital Hamilton": "Robert Wood Johnson University Hospital Hamilton",
    "RWJ University Hospital Rahway": "Robert Wood Johnson University Hospital Rahway",
    "RWJUH Somerset": "Robert Wood Johnson University Hospital Somerset",
    "RWJUH Hamilton": "Robert Wood Johnson University Hospital Hamilton",
    "RWJUH Rahway": "Robert Wood Johnson University Hospital Rahway",
    "Rutgers Cancer Institute": "Rutgers Cancer Institute of New Jersey",
    "Cancer Institute of New Jersey": "Rutgers Cancer Institute of New Jersey",
    "RCINJ": "Rutgers Cancer Institute of New Jersey",
    "RWJBarnabas Health Medical Group": "Barnabas Health Medical Group",
    "Barnabas Medical Group": "Barnabas Health Medical Group",
    "BHMG": "Barnabas Health Medical Group",
    "Childrens Specialized Hospital": "Children's Specialized Hospital",
    "CSH": "Children's Specialized Hospital",
    "RWJ Barnabas Health School of Nursing": "RWJBarnabas Health School of Nursing",
    "Multiple Hospital": "Multiple Hospitals",
    "Multi Hospital": "Multiple Hospitals",
    "Multiple Sites": "Multiple Hospitals",
    "Saint Barnabas Medical Center": "Cooperman Barnabas Medical Center",
    "St Barnabas Medical Center": "Cooperman Barnabas Medical Center",
    "St. Barnabas Medical Center": "Cooperman Barnabas Medical Center",
}

# The review cell maintains its own list so you can run it independently.
OFFICIAL_NAMES = [
    "Clara Maass Medical Center", "Community Medical Center",
    "Cooperman Barnabas Medical Center", "Jersey City Medical Center",
    "Monmouth Medical Center", "Monmouth Medical Center Southern Campus",
    "Newark Beth Israel Medical Center", "Robert Wood Johnson University Hospital",
    "Robert Wood Johnson University Hospital Somerset",
    "Robert Wood Johnson University Hospital Hamilton",
    "Robert Wood Johnson University Hospital Rahway",
    "Trinitas Regional Medical Center", "Rutgers Cancer Institute of New Jersey", "RWJBarnabas Health School of Nursing",
    "Children's Specialized Hospital", "Barnabas Health Medical Group",
    "Multiple Hospitals", "Other",
]
names = {clean_hospital_name(name): name for name in OFFICIAL_NAMES}
for typed, canonical in MANUAL_ALIASES.items():
    names[clean_hospital_name(typed)] = canonical

# The spelling checker is cautious: it acts only on a single very close match.
def resolve_hospital(typed, excel_row):
    key = clean_hospital_name(typed)
    if key in names:
        return names[key]
    matches = get_close_matches(key, names.keys(), n=2, cutoff=0.88)
    if len(matches) == 1:
        corrected = names[matches[0]]
        print(f"Excel row {excel_row}: '{typed}' → '{corrected}'")
        return corrected
    raise ValueError(
        f"Excel row {excel_row}: I cannot confidently identify '{typed}'. "
        "Please enter a clearer hospital/institution name in Excel."
    )

approved_rows = []
for row_number, (_, article) in enumerate(review.iterrows(), start=2):
    decision = str(article["Include?"]).strip().casefold()
    if decision not in {"", "yes", "y", "no", "n"}:
        raise ValueError(f"Excel row {row_number}: Include? should be Yes or No (or blank).")
    if decision not in {"yes", "y"}:
        continue
    hospitals = [h.strip() for h in str(article["Hospital to Assign"]).split(";") if h.strip()]
    if not hospitals:
        raise ValueError(f"Excel row {row_number}: Enter a hospital for this approved article.")
    hospitals = [resolve_hospital(h, row_number) for h in hospitals]
    record = article.to_dict()
    record["Matched Hospitals"] = "; ".join(dict.fromkeys(hospitals))
    record["Status"] = "Approved after review"
    approved_rows.append(record)

combined = pd.concat([original, pd.DataFrame(approved_rows)], ignore_index=True)
combined = combined.drop_duplicates(subset="PMID", keep="last")
# Any article linked to two or more hospitals appears only once in
# the Multiple Hospitals section. A manual 'Multiple Hospitals' entry also works.
def publication_category(value):
    hospitals = [h.strip() for h in str(value).split(";") if h.strip()]
    return "Multiple Hospitals" if len(set(hospitals)) > 1 else (hospitals[0] if hospitals else "")

combined["Hospital"] = combined["Matched Hospitals"].map(publication_category)
organized = combined[combined["Hospital"].ne("")].copy()
organized["_sort_date"] = organized["Publication Date"].map(date_for_sort)
def section_rank(category):
    if category == "Barnabas Health Medical Group":
        return 1
    if category == "Multiple Hospitals":
        return 2
    if category == "Other":
        return 3
    if category == "Rutgers Cancer Institute of New Jersey":
        return 4
    if category == "RWJBarnabas Health School of Nursing":
        return 5
    return 0
organized["_section_last"] = organized["Hospital"].map(section_rank)
organized = organized.sort_values(
    ["_section_last", "Hospital", "_sort_date", "Title"],
    ascending=[True, True, False, True], na_position="last", kind="stable"
)


# Word updates PAGE and NUMPAGES fields when opening/printing the document.
def add_page_number_footer(document):
    from docx.enum.text import WD_ALIGN_PARAGRAPH
    from docx.oxml import OxmlElement
    from docx.oxml.ns import qn
    paragraph = document.sections[0].footer.paragraphs[0]
    paragraph.alignment = WD_ALIGN_PARAGRAPH.CENTER
    paragraph.paragraph_format.space_before = Pt(0)
    paragraph.paragraph_format.space_after = Pt(0)
    paragraph.add_run("Page ")
    def field(name):
        run = paragraph.add_run()
        begin = OxmlElement("w:fldChar")
        begin.set(qn("w:fldCharType"), "begin")
        instr = OxmlElement("w:instrText")
        instr.set(qn("xml:space"), "preserve")
        instr.text = f" {name} "
        separate = OxmlElement("w:fldChar")
        separate.set(qn("w:fldCharType"), "separate")
        placeholder = OxmlElement("w:t")
        placeholder.text = "1"
        end = OxmlElement("w:fldChar")
        end.set(qn("w:fldCharType"), "end")
        for node in (begin, instr, separate, placeholder, end):
            run._r.append(node)
    field("PAGE")
    paragraph.add_run(" of ")
    field("NUMPAGES")

final_doc = Document()
sec = final_doc.sections[0]
sec.top_margin = Inches(PAGE_MARGIN_INCHES)
sec.bottom_margin = Inches(PAGE_MARGIN_INCHES)
style = final_doc.styles["Normal"]
style.font.name = FONT_NAME
style.font.size = Pt(FONT_SIZE)
style.paragraph_format.space_before = Pt(0)
style.paragraph_format.space_after = Pt(0)
style.paragraph_format.line_spacing = 1.0

def add_line(value, bold=False):
    paragraph = final_doc.add_paragraph(str(value))
    paragraph.paragraph_format.space_before = Pt(0)
    paragraph.paragraph_format.space_after = Pt(0)
    paragraph.paragraph_format.line_spacing = 1.0
    if bold and paragraph.runs:
        paragraph.runs[0].bold = True

def add_clickable_link(paragraph, label, url):
    """Add a real clickable Word hyperlink (not merely URL text)."""
    from docx.oxml import OxmlElement
    from docx.oxml.ns import qn
    from docx.opc.constants import RELATIONSHIP_TYPE as RT
    relation_id = paragraph.part.relate_to(str(url), RT.HYPERLINK, is_external=True)
    hyperlink = OxmlElement("w:hyperlink")
    hyperlink.set(qn("r:id"), relation_id)
    run = OxmlElement("w:r")
    properties = OxmlElement("w:rPr")
    color = OxmlElement("w:color")
    color.set(qn("w:val"), "0563C1")
    underline = OxmlElement("w:u")
    underline.set(qn("w:val"), "single")
    properties.append(color)
    properties.append(underline)
    run.append(properties)
    text = OxmlElement("w:t")
    text.text = str(label)
    run.append(text)
    hyperlink.append(run)
    paragraph._p.append(hyperlink)

def add_link_line(document, prefix, label, url):
    paragraph = document.add_paragraph()
    paragraph.paragraph_format.space_before = Pt(0)
    paragraph.paragraph_format.space_after = Pt(0)
    paragraph.paragraph_format.line_spacing = 1.0
    paragraph.add_run(prefix)
    add_clickable_link(paragraph, label, url)
    return paragraph

# Overview table: count each publication once in its assigned category.
# Only display institutions with publications this month.
def add_summary_table(document, organized_frame, institution_names):
    from docx.enum.table import WD_TABLE_ALIGNMENT, WD_CELL_VERTICAL_ALIGNMENT
    from docx.enum.text import WD_ALIGN_PARAGRAPH
    from docx.shared import Pt

    counts = organized_frame["Hospital"].value_counts().to_dict()
    clinical = sorted(set(institution_names) - {
        "Barnabas Health Medical Group", "Multiple Hospitals", "Other",
        "Rutgers Cancer Institute of New Jersey", "RWJBarnabas Health School of Nursing"
    })
    categories = clinical + ["Barnabas Health Medical Group", "Multiple Hospitals", "Other",
                             "Rutgers Cancer Institute of New Jersey", "RWJBarnabas Health School of Nursing"]
    title = document.add_paragraph()
    title.add_run("Publications by institution").bold = True
    title.paragraph_format.space_before = Pt(8)
    title.paragraph_format.space_after = Pt(4)

    table = document.add_table(rows=1, cols=2)
    table.style = "Light Shading Accent 1"
    table.alignment = WD_TABLE_ALIGNMENT.CENTER
    table.cell(0, 0).text = "Institution / Category"
    table.cell(0, 1).text = "Articles"
    for category in categories:
        # Only show institutions with at least one publication this month.
        if counts.get(category, 0) == 0:
            continue
        cells = table.add_row().cells
        cells[0].text = category
        cells[1].text = str(counts[category])
    total = table.add_row().cells
    total[0].text = "Total unique publications"
    total[1].text = str(len(organized_frame))
    for cell in table.rows[0].cells + table.rows[-1].cells:
        for paragraph in cell.paragraphs:
            for run in paragraph.runs:
                run.bold = True
    for row in table.rows:
        row.cells[1].vertical_alignment = WD_CELL_VERTICAL_ALIGNMENT.CENTER
        for p in row.cells[1].paragraphs:
            p.alignment = WD_ALIGN_PARAGRAPH.RIGHT
        for cell in row.cells:
            for p in cell.paragraphs:
                p.paragraph_format.space_after = Pt(0)
                p.paragraph_format.space_before = Pt(0)
    below = document.add_paragraph()
    below.paragraph_format.space_after = Pt(6)
    return table

final_doc.add_heading(NEWSLETTER_TITLE, level=0)
heading = final_doc.add_paragraph(month_start.strftime("%B %Y"))
heading.paragraph_format.space_after = Pt(10)

add_summary_table(final_doc, organized, OFFICIAL_NAMES)

for hospital, papers in organized.groupby("Hospital", sort=False):
    h = final_doc.add_heading(f"{hospital} ({len(papers)})", level=1)
    h.paragraph_format.space_before = Pt(12)
    h.paragraph_format.space_after = Pt(6)
    for number, (_, article) in enumerate(papers.iterrows(), start=1):
        add_line(f"{number}. {article['Title']}", bold=True)
        add_line(f"Authors: {article['Authors'] or 'Not listed'}")
        add_line(f"Journal: {article['Journal']} | Publication date: {article['Publication Date'] or 'Not stated'}")
        add_link_line(final_doc, "PubMed: ", article["PubMed Link"], article["PubMed Link"])
        if article["DOI"]:
            add_link_line(final_doc, "DOI: ", article["DOI"], f"https://doi.org/{article['DOI']}")
        spacer = final_doc.add_paragraph()
        spacer.paragraph_format.space_after = Pt(0)

# Create a new final version on every run; never overwrite an earlier final.
base_stem = review_excel.stem
for final_number in range(1, 1000):
    final_path = OUTPUT_FOLDER / f"{base_stem}_FINAL_v{final_number:02d}.docx"
    if not final_path.exists():
        break
else:
    raise RuntimeError("Too many final revisions.")
add_page_number_footer(final_doc)
final_doc.save(final_path)
print(f"Approved {len(approved_rows)} additional publications.")
print(f"Final Word document: {final_path.resolve()}")


Reading reviewed Excel: RWJBH_Publications_2026-09_2026-10-15_v01.xlsx
Approved 1 additional publications.
Final Word document: /Users/richardsavel/Library/CloudStorage/Dropbox/python test files/pubmed newsletter/chat version/RWJBH_Publications_2026-09_2026-10-15_v01_FINAL_v02.docx
